# Session 14 · Prompt anatomy
## The same model, a very different result — and a prompt library you keep in Git

**Role · task · context · constraints · output format · examples.** Two jobs today. First, learn what a prompt is made of, and *measure* whether a change made it better. Second, create **your own work repository** and push your first prompts to it. Prompts are assets: they get versions, tests, and a history.

**Before starting:** `git pull`, `uv sync` (no new packages). Create your work repository first if you have not (`module02/work_repo_steps.md`).

**How to use this notebook.** It has three parts.

| Part | What it is | Code? |
|---|---|---|
| **Part A · Read before you run** | Why this topic matters, the words you need, and how today's session is built | No code. Read it first (about 15 minutes). |
| **Part B · Hands-on** | Sections 1 to 8. Every section follows the same six steps (below). | Yes |
| **Part C · Wrap-up** | Summary, interview questions, troubleshooting | No |

Every section in Part B follows the same six steps, so you always know where you are:

1. **What and why** — what we are about to do and why it matters.
2. **The idea** — the concept, with an analogy and often a small diagram.
3. **What the next cell does** — a numbered walk-through *before* you see the code.
4. **The code** — run it.
5. **What you should see** — how to read the output, and what it means if yours is different.
6. **Check yourself** — one short question and a common mistake to avoid.

Boxes marked **▸ Read more** are optional. Click to open them when you want the deeper background; you can skip them in class.

---
# Part A · Read before you run

## A1. Why this session matters

### The problem, in one story
A retail bank wants to save its staff time. A developer connects a strong model to four daily jobs, each with a one-line prompt:

| Job | The one-line prompt |
|---|---|
| EMI reminders by SMS | "Write a reminder." |
| Credit-committee summaries | "Summarise this applicant." |
| "Application declined" SMS | "Write a message to the customer about their loan." |
| Complaint triage for the ticketing system | "What is this complaint about?" |

It is the same model the bank tested last week, and the demo looked fine. In the first month of real use:

1. **Some reminders threaten customers.** A few messages say *"to avoid late fees and legal action"*. Nobody asked for that, and nobody told the model not to. Customers complain; the bank's own fair-treatment rules forbid that tone.
2. **A summary recommends the wrong decision.** An applicant with one missed payment is marked "a solid candidate, approve". The bank's lending policy says *refer to a senior underwriter*. The model never saw the policy. It could not follow a rule it was never shown.
3. **A declined customer learns a secret.** The SMS says *"your internal risk score of 612 was below our cutoff of 650"*. That number was in the data the model received, and nothing said it was confidential.
4. **The ticketing integration breaks.** The triage replies are friendly sentences, or JSON wrapped in a Markdown code fence. The ticketing system expects bare JSON. Every malformed reply becomes a manual ticket.
5. **Then someone "fixes" a prompt.** Things get better, then worse. Nobody can say which version was running on which day, or why a line was added.

None of these is a problem with the model. **They are problems with what the model was told, and with how the prompts were managed.** These five failures are exactly the four tasks of today's lab plus the library you build at the end.

### What you learn today, and what each one prevents

| You learn to... | It prevents... | The everyday name |
|---|---|---|
| Write a prompt in **five parts**: role, task, context, constraints, output format | The model guessing the audience, the length, the facts and the tone | A proper brief, not a sticky note |
| Put **standing rules in the system message** and **the job in the user message** | Rules that get lost, repeated or mixed with data | Standing orders on the wall vs the job slip on the desk |
| Add **worked examples** (few-shot) when the rule is your own | "Obvious" in-house rules that the model cannot know | Learning from three filled-in forms |
| Write **acceptance checks** first, then improve the prompt until they pass | "It looks better to me" arguments with no number | Acceptance criteria on a ticket |
| Keep every version in a **prompt library** in **your own Git repository** | Lost history, untraceable edits, no submission | The prompt's history book |

### Where this is used in industry
- **Banking (BFS):** drafting credit memos from application data, writing customer letters and SMS that must include some facts and never reveal others, routing complaints into categories that downstream systems read.
- **Healthcare:** turning a clinician's notes into a fixed-format summary. The format (sections, length) and the constraints ("never add a diagnosis that is not in the notes") are written into the prompt.
- **Insurance:** reading claims emails and returning JSON with a claim type and urgency, so a workflow engine can pick it up without a person.
- **Retail and telecom:** support replies and ticket triage at high volume, where one house rule ("refund questions go to billing, not orders") must be taught to the model.

In all of these, teams keep prompts in version control (or a prompt registry) next to the code, and test them before a change goes live. Today you start doing the same, on a small scale.

### Why interviewers care
These questions come up in AI-engineer and GenAI-developer interviews:
- *"What makes a good prompt? Walk me through its parts."*
- *"What goes in the system message and what goes in the user message?"*
- *"When would you use few-shot prompting instead of zero-shot, and what does it cost?"*
- *"How do you know that a prompt change actually made things better?"*
- *"How do you manage prompts in a team?"*

By the end of today you can answer each one with a definition, an example from your own lab, and one honest limit. Part C has model answers.

<details><summary><b>▸ Read more: "Just make the model bigger" — why prompting comes first</b></summary>

Sessions 9 to 13 were about the model: what it sees, which one to pick, where it runs. Changing the model is expensive: new costs, new limits, new tests, sometimes a new contract. Changing the words you send costs nothing and takes minutes. That is why the professional order is: (1) write a clear prompt and measure it, (2) add examples, (3) only then consider a different model, retrieval (Module 5) or fine-tuning (Session 54). A bigger model still cannot follow a policy it has never been shown, and it still cannot guess that *your* bank files UPI problems under "account".
</details>

## A2. Where this session fits in the course

```
 S12  Local and open models ─────► the same client talks to a model on your laptop
  │
 S13  Multimodal + structured ───► JSON that matches a schema; "asking for JSON" vs "guaranteeing it"
  │
 S14  Prompt anatomy ────────────► TODAY: five parts, few-shot, acceptance checks, prompt library,
  │                                and YOUR OWN work repository (first push, first submission)
 S15  Reasoning prompts ─────────► think step by step? Reuses today's complaint classifier; your
  │                                library gets version 2 entries (so never edit version 1)
 S16  Structured prompting ──────► today's one-line classifier instruction becomes the "baseline";
  │                                tags, templates, a system prompt kept as a file in your repo
 S17  Prompt evaluation ─────────► today's acceptance checks grow into a golden set and a judge
  │
 S18  Defensive prompting ───────► attackers try to break constraints like "never reveal"
  │
 S20  Function calling ──────────► "output format" becomes a tool's argument schema
 S28+ RAG (Module 5) ────────────► "context" is fetched automatically instead of pasted by you
 S52, S54  Evaluations, LLMOps ──► today's checks and library at team scale (A/B tests of prompts)
```

**What you need from earlier sessions:**
- A working `.env` in the course folder with `BASE_URL`, `API_KEY` and `MODEL` (Sessions 2 and 11). Without it, the model cells print "skipped"; everything else still runs.
- The course repository pulled and synced (`git pull`, `uv sync`).
- Git installed and a **GitHub account with a verified email** (Session 1 or the Git webinar). You need your GitHub username today.
- The idea of a **messages list** from Sessions 2 and 11: a model call is a list of `{"role": ..., "content": ...}` items. Today you only change what goes into it.

## A3. The words you need today

Read these once now. Each term appears in bold the first time it is used in Part B.

---
**1. Prompt**
- **Definition:** everything you send to the model for one request: the instructions, the facts it needs, any examples, and the input itself.
- **Analogy:** the brief you give a very capable temp who started this morning and knows nothing about your bank.
- **Example:** today's weak prompt is the four words "Write a reminder." The strong prompt is about eighty words in five parts.
- **Often confused with:** the user's question only. The question is one piece; the prompt is the whole package.
- **Comes back in:** every session from here.

---
**2. System message and user message**
- **Definition:** two roles in the messages list. The **system** message holds standing rules that apply to every request. The **user** message holds this request's job and its input.
- **Analogy:** the standing orders pinned to the wall (system) and the job slip on today's desk (user).
- **Example:** "Use at most 35 words" goes in the system message; "Customer: Sneha Iyer, EMI Rs 18,500, due 5 November" goes in the user message.
- **Often confused with:** a security boundary. The system message carries more weight, but it does not make the model obey perfectly (Session 18).
- **Comes back in:** S16, S19, Module 6.

---
**3. Role**
- **Definition:** the part of the prompt that says who the model is and who it is writing for.
- **Analogy:** telling the temp "you are writing as the bank's customer-service team, to a customer".
- **Example:** "You are a credit analyst writing for a credit committee." It sets vocabulary and level.
- **Often confused with:** the message *role* (`system`, `user`, `assistant`). Same word, different meaning: one is a persona, the other is a slot in the messages list.
- **Comes back in:** S15, S16.

---
**4. Task**
- **Definition:** what to do, in one sentence that starts with a verb.
- **Analogy:** the subject line of a work ticket.
- **Example:** "Summarise the applicant and recommend approve, refer or decline."
- **Often confused with:** the whole prompt. A task alone ("Write a reminder.") is exactly the weak prompt.
- **Comes back in:** every session.

---
**5. Context**
- **Definition:** facts and rules the model *cannot know* unless you supply them: your policy text, your definitions, the situation.
- **Analogy:** the folder you hand the temp along with the job: the policy, the customer file.
- **Example:** the five-rule lending policy for Task 1.
- **Often confused with:** constraints. Context is *information* ("Rule 4: refer if exactly one missed payment"); constraints are *limits on the answer* ("at most 70 words").
- **Comes back in:** Module 5 (RAG fetches context automatically), S38 (context engineering).

---
**6. Constraints**
- **Definition:** limits the answer must respect: length, what must be included, what must never appear.
- **Analogy:** the fence around the answer.
- **Example:** "Never reveal internal scores, cutoffs or the internal reason."
- **Often confused with:** a guarantee. A constraint makes the right answer much more likely; only a check can confirm it.
- **Comes back in:** S16, S18 (defensive prompting).

---
**7. Output format**
- **Definition:** the exact shape of the reply.
- **Analogy:** the form the answer must be written on.
- **Example:** "A single JSON object with exactly the keys category and urgency, and nothing else: no prose, no markdown code fences."
- **Often confused with:** constraints. They overlap; the useful split is "how big and what to avoid" (constraints) versus "what shape" (format).
- **Comes back in:** S13 (schemas), S16, S20 (tool arguments).

---
**8. Zero-shot, one-shot, few-shot**
- **Definition:** a prompt with no worked examples, one example, or several examples (input and correct output).
- **Analogy:** explaining a form to a new joiner (zero), showing one filled-in form (one), showing three or four (few).
- **Example:** the classifier with 0, 1 and 4 examples in section 4.
- **Often confused with:** training. The model's weights do not change; the examples are just part of the prompt, and you pay for them as tokens on every call.
- **Comes back in:** S15, S16, S17.

---
**9. House convention**
- **Definition:** a rule that is obvious inside your organisation but invisible to the model.
- **Analogy:** "we always file these under X" — the thing every colleague knows and no manual states clearly.
- **Example:** this bank files a failed UPI payment under **account**, and a request to change a mobile number under **other**.
- **Often confused with:** common sense. A model's "common sense" might file a UPI failure under *card* or *other*. Only your examples or your context tell it otherwise.
- **Comes back in:** S16, S17.

---
**10. Acceptance check**
- **Definition:** a few lines of Python that read an output and return pass or fail, with a short detail.
- **Analogy:** the acceptance criteria on a ticket: the work is "done" when each one is ticked.
- **Example:** `max_words(40)`, `has("18,500")`, `has_none_of(["penalty", "late fee", "legal"])`, `json_only()`.
- **Often confused with:** a full quality judgement. Checks catch what a machine can catch (length, facts, forbidden words, shape). They cannot judge tone, truth or taste.
- **Comes back in:** S17 (golden sets), S52 (evaluation suites).

---
**11. Baseline**
- **Definition:** the simple version you compare against, so you can say "better than what?".
- **Analogy:** the "before" photo.
- **Example:** each task's weak prompt. And the one-line classifier instruction from section 4 becomes the official baseline in Session 16.
- **Comes back in:** S16, S17.

---
**12. Prompt library and versioning**
- **Definition:** a file (`prompts/library.md`) with one entry per prompt version: the prompt's parts, the output it produced, and which checks passed. *Versioning* means you never edit an old entry; you add version 2.
- **Analogy:** a lab notebook: you never rub out an old result, you write the new one below it.
- **Example:** `## T2-sms-declined | v2 | 2026-10-06`
- **Often confused with:** a folder of "final" prompts. A library keeps the failures too, because they explain why the current version says what it says.
- **Comes back in:** S15 (version 2 entries), S16–S18, S54.

---
**13. Work repository (versus the course repository)**
- **Definition:** your own GitHub repository, `genai-course-work`, where you commit and push your work. The **course** repository, `genai-agents-course`, is where you only pull.
- **Analogy:** the course repository is a library book the instructor keeps updating; your work repository is your own notebook.
- **Example:** `Documents\genai-agents-course\` (pull) and `Documents\genai-course-work\` (push), side by side.
- **Often confused with:** a fork or a sub-folder. It is a separate repository. Never put one inside the other.
- **Comes back in:** every submission from now on.

---
**14. Commit, push, remote (origin)**
- **Definition:** a **commit** records a labelled snapshot of your files on your laptop. A **push** sends your commits to GitHub. The **remote** called `origin` is the GitHub copy your folder is linked to.
- **Analogy:** commit = save a version in your notebook; push = hand it in.
- **Example:** `git commit -m "Session 14: prompt library v1"` then `git push -u origin main`.
- **Often confused with:** each other. A commit alone never leaves your laptop; the instructor sees nothing until you push.
- **Comes back in:** every submission.

---
**15. `.gitignore` and "tracked"**
- **Definition:** a file is **tracked** when Git records it in commits. `.gitignore` lists files Git should leave alone, such as `.env`.
- **Analogy:** a "do not photocopy" stamp.
- **Example:** the notebook creates a `.gitignore` containing `.env`; the self-check confirms `.env` is not tracked.
- **Often confused with:** deleting. Ignored files stay on your laptop; they just never go into a commit. And `.gitignore` does not un-track a file that was already committed.
- **Comes back in:** every submission; S55 (secrets in the OWASP Top 10).

## A4. How this session is built

### The big picture

```
┌─── COURSE REPOSITORY (you pull it; you do not edit it)  ─────────────────────────────────────────────────────┐
│                                                                                                              │
│ module02/prompt_tasks.py     THE TASKS AND THEIR CHECKS                                                      │
│    DEMO   EMI reminder (instructor demo)      TASK1  credit-committee summary (+ lending policy)             │
│    TASK2  'declined' SMS that must not leak   TASK3  complaint triage as bare JSON                           │
│    each task: a weak prompt, the input, material you may use, ACCEPTANCE CHECKS, a hidden reference          │
│    EXAMPLES = 4 worked examples   COMPLAINTS_TEST = 8 labelled messages   classify_spec · score_classifier   │
│                                                                                                              │
│ module02/prompt_utils.py     THE TOOLS                                                                       │
│    PromptSpec (five parts + examples -> messages list) · show_tips · results_table                           │
│    init_work_repo · save_to_library (writes YOUR library) · check_work_repo (nine plain-English checks)      │
│                                                                                                              │
│ module01/providers_utils.py (S11)   reads your .env and makes the client (groq_client)                       │
│                                                                                                              │
│ module02/s14_prompt_anatomy.ipynb   <── YOU ARE HERE                                                         │
│ tests/test_s14.py                   the checkpoint test (offline; 4 live tests skip without a key)           │
│ module02/work_repo_steps.md         the Git steps and what each error means (keep it open)                   │
└──────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
                         │ the notebook writes prompts/library.md and reads your repository ▼
┌─── YOUR SIDE (you own it)  ──────────────────────────────────────────────────────────────────────────────────┐
│ course folder/.env                     BASE_URL, API_KEY, MODEL. Git-ignored. Never copy it anywhere.        │
│ Documents/genai-course-work/           YOUR work repository, cloned from YOUR GitHub (Public)                │
│    prompts/library.md                  one entry per prompt version: parts, output, checks                   │
│    .gitignore  README.md               created for you if missing (never overwritten)                        │
│ GitHub: github.com/<you>/genai-course-work   what your instructor reads after you push                       │
└──────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
```

### What happens to one prompt

```
   YOU fill in PromptSpec                         the task (from prompt_tasks.py)
   role · task · context ·                        user_input  +  acceptance checks
   constraints · output_format                          │                │
          │                                             │                │
          ▼  show_tips: "Missing parts: ..."            │                │
          │                                             │                │
          ▼  to_messages(user_input)  ◄─────────────────┘                │
   [ {system: role + "Rules:" + "Output format:"},                       │
     {user:   "Task:" + "Context:" + "Input:"} ]                         │
          │                                                              │
          ▼  chat(): client.chat.completions.create(temperature=0)       │
      the model  (Groq, MODEL from .env)                                 │
          │                                                              │
          ▼  reply text ───────────────────────────► run_checks ◄────────┘
                                                        │
                                       PASS / FAIL per check, "-> 3/5 checks passed"
                                                        │
                        all pass? ── no ──► change ONE part, run again
                                   └─ yes ─► save_to_library (v1) ─► git commit ─► git push
                                                                                      │
                                                                     check_work_repo: all PASS
```

### The helpers you will call, and what each does

| Helper | Lives in | You give it | It gives back | Used in section |
|---|---|---|---|---|
| `groq_client()` | providers_utils (S11) | nothing (it reads `.env`) | an OpenAI-compatible client | 1 |
| `PromptSpec(role=, task=, context=, constraints=, output_format=, examples=)` | prompt_utils | the parts of your prompt | an object that holds them | 3, 5 |
| `spec.to_messages(user_input)` | prompt_utils | the input text | the messages list that is sent | 3 |
| `show_tips(spec)` | prompt_utils | a spec | prints which of the five parts are empty, with a tip for each | 3, 5 |
| `results_table(results)` | prompt_utils | check results | a printable PASS/FAIL table with "-> n/m checks passed" | 2, 5 |
| `task.weak_spec()`, `task.reference`, `task.acceptance()` | prompt_tasks | — | the weak prompt as a spec, the instructor's spec, the check names | 2, 3, 5 |
| `run_task(client, MODEL, task, spec)` | prompt_tasks | a client, the model name, a task, your spec | (reply text, list of check results): one model call | 2, 5 |
| `run_checks(task, text)` | prompt_tasks | a task and any text | the check results, with no model call | 2 (under the hood) |
| `classify_spec(n)` | prompt_tasks | 0, 1 or 4 | the classifier spec with n worked examples | 4 |
| `score_classifier(client, MODEL, spec)` | prompt_tasks | a client, model, classifier spec | right labels, exact-format count, the wrong ones: 8 model calls | 4 |
| `read_label(reply)` | prompt_tasks | a reply | (label or None, was it exactly one word?) | 4 (under the hood) |
| `init_work_repo(path)` | prompt_utils | your work repo folder | creates `.gitignore`, `README.md`, `prompts/library.md` if missing; returns their names | 6 |
| `save_to_library(path, id, version, spec, output, results, purpose=)` | prompt_utils | one prompt version and its results | appends one entry to `prompts/library.md`; never edits old ones | 6 |
| `check_work_repo(path)` | prompt_utils | your work repo folder | nine PASS/FAIL checks, each failure with its exact fix | 7 |
| `card(task)`, `show(task, spec, label)` | **this notebook** (section 2) | a task (and a spec) | prints the task card; runs one prompt and prints the reply and checks | 2, 5 |

You do not need to read the helper files, and for `prompt_tasks.py` please **don't**: the instructor's reference prompts are in there, and the learning is in trying first. Where a helper matters, an **under the hood** cell rebuilds a short version of it in front of you and checks that it gives the same answer as the real one.

### The order of the sections

| # | Section | Calls the model? | Minutes |
|---|---|---|---|
| 1 | Setup | no | 2 |
| 2 | Same model, two prompts | yes, 2 calls | 5 |
| 3 | The parts, and where each one goes | no | 4 |
| 4 | Zero-shot, one-shot, few-shot | yes, 24 calls | 6 |
| 5 | Your turn: three real prompts | yes, 2 calls per task, then 1 per try | 24 |
| 6 | Your prompt library — a versioned asset | no | 3 |
| 7 | Commit and push — your first submission | no (Git in a terminal) | 9 |
| 8 | Reflection | no | 3 |

## A5. What you will be able to do, and what you will submit

**By the end of this session you will be able to:**
1. **Name** the five parts of a prompt and **identify** which ones a weak prompt is missing.
2. **Place** each part in the right message: standing rules in the system message, the job in the user message, examples as earlier turns.
3. **Compare** zero-, one- and few-shot prompts on the same eight messages, and **decide** whether examples are worth their tokens.
4. **Rewrite** three weak prompts until every acceptance check passes, changing one part at a time.
5. **Record** each prompt version with its output and check results in a library, without editing old entries.
6. **Create, commit and push** your own work repository, and **verify** it with the self-check.

**What you submit (in your work repository, `genai-course-work`):**

| File | What it must contain |
|---|---|
| `prompts/library.md` | Entries for **three different tasks** (`T1-credit-summary`, `T2-sms-declined`, `T3-complaint-triage`), written by the save cell in section 6 |
| `.gitignore` | A line that is exactly `.env` (created for you) |
| Everything committed and pushed | GitHub shows `prompts/library.md`; nothing left uncommitted or unpushed |

**The checkpoint:** every line of the self-check in section 7 says `PASS`, and `uv run pytest tests/test_s14.py` is green in the course repository. Then submit your repository link through the form.

Be precise about what the test proves. `tests/test_s14.py` checks the course's helpers and acceptance checks offline (25 tests, including real Git repositories in a temporary folder). Its 4 **live** tests send the instructor's reference prompts to the model, and they are **skipped** without a key. *A skipped test is not a pass.* Your own work is proved by the self-check and by the file on GitHub.

**Budget.** This session has no answer cache and no token meter (Session 15 adds both), so every run of a model cell calls the model again. Section 4's 24 calls may each reply with at most 300 tokens (`max_tokens=300`); a task call may reply with at most 1,500. The free Groq plan allows about 8,000 tokens a minute and 200,000 a day for this model, so the whole lab fits comfortably if you do not rerun section 4 in a loop. If you see a 429 error, wait 30 seconds and rerun only that cell.

---
# Part B · Hands-on

## 1. Setup

### What and why
Before anything else, the notebook must find the course's helper files, read your `.env`, and connect to the model. If something is missing, it should tell you *now*, in plain words, rather than halfway through the lab.

### The idea
Think of this as opening the branch in the morning: unlock the door (find the files), switch on the system (read your key), and check the counter is ready (make the client). If the system is down, the branch still opens: you can do the paperwork while you wait. In this notebook, that "paperwork" is real learning: the five parts, the messages list, the checks, the library and the Git steps all work without a key.

### What the next cell does
1. **`sys.path[:0] = [...]`** puts the course folders at the *front* of Python's search list. `[:0]` is a **slice** meaning "before the first item", so this inserts all five folders at once. Now `import prompt_utils` finds the file whether you opened the notebook from the repo root or from inside `module02/`.
2. **`from pathlib import Path`** brings in `Path`, Python's way to handle file paths that works on Windows and Mac alike. Section 6 uses it for your work repository.
3. **`try: ... except ImportError as e:`** runs the imports, and *if* one fails, catches the error and raises a clearer one that tells you what to do. **`raise ... from e`** keeps the original error attached underneath, so nothing is hidden.
4. The imports bring in the Session 11 client maker and today's two files. **Importing `providers_utils` reads your `.env`** (it calls `load_dotenv(override=True)`), so `os.getenv(...)` sees your settings from here on.
5. **`client = None`**, then `client = groq_client()` only if `BASE_URL` and `API_KEY` exist and the key is not the placeholder `paste_your_...` from `.env.example`. **`ready`** is then `True` or `False`, and every model cell checks it.
6. Prints the model name, and a one-line note if there is no key.

<details><summary><b>▸ Read more: why not just stop if there is no key?</b></summary>

Because more than half of today does not need the model. Section 3 (the parts and the messages list), the task cards, `show_tips`, the library and the Git self-check are pure Python. A student whose key is missing, or who has used up the day's tokens, should still learn those. So model cells print "skipped" instead of crashing. **Skipped is never the same as passed**: the live tests report a skip clearly, and your library needs real outputs.
</details>

In [ ]:
import sys; sys.path[:0] = [".", "module01", "module02", "../module01", "../module02"]   # find the course files from either folder
import os
from pathlib import Path
try:
    from providers_utils import groq_client                                   # Session 11 (importing it reads .env)
    from prompt_utils import (PromptSpec, check_work_repo, init_work_repo, results_table, save_to_library, show_tips)
    from prompt_tasks import (DEMO, TASK1, TASK2, TASK3, classify_spec, run_task, score_classifier)
except ImportError as e:
    raise ImportError(f"{e}. This notebook uses files from Sessions 11 and 14: run 'git pull' in the course folder, "
                      "then restart the kernel.") from e

client = None                                                                 # stays None when there is no key
if os.getenv("BASE_URL") and os.getenv("API_KEY") and "paste_your" not in os.getenv("API_KEY", ""):
    client = groq_client()
ready = client is not None                                                    # every model cell checks this
MODEL = os.getenv("MODEL")
print("model:", MODEL)
if not ready:
    print("No key in .env: the cells that call the model will print 'skipped'. Everything else still runs.")

### What you should see
With a key, one line:
```
model: openai/gpt-oss-20b
```
(That is the model pinned in `.env.example`; the instructor may pin a different one for your batch.) Without a key, two lines: `model: None` (or the model name, if `MODEL` is set) and the "No key in .env" note.

| If you see... | It means... | Do this |
|---|---|---|
| `ImportError ... run 'git pull'` | a course file is missing | `git pull`, `uv sync`, then **Kernel → Restart** |
| `model: None` and "No key" | `.env` was not found, or has no `API_KEY` | `.env` must be in the course folder (`copy .env.example .env`), with your key pasted after `API_KEY=` |
| "No key" but you pasted a key | the key still contains `paste_your`, or you edited `.env` after starting the kernel | fix `.env`, then **Kernel → Restart** and run again |

### Check yourself
*Why does the notebook still run without errors when there is no key? Which sections will still teach you something?*

> **Common mistake:** editing `.env` and re-running only this cell. Python has already read the old values. Restart the kernel after changing `.env`.

## 2. Same model, two prompts

### What and why
Nothing about the model changes in this section. Only the words we send. Each task has **acceptance checks**: small pieces of Python that say whether an output is good enough, like the acceptance criteria on a ticket. They turn "this looks better" into a number.

The task is the bank's EMI reminder. The input is three facts about a customer. You send the **same model** the **same input** twice: first with a four-word prompt, then with a prompt that has all five parts. Then you read the checks.

### The idea: a brief versus a sticky note
```
 WEAK PROMPT (a sticky note)                 STRONG PROMPT (a proper brief)
 ──────────────────────────────────          ─────────────────────────────────────────────────
 "Write a reminder."                         Role:        customer-service writer for a bank
                                             Task:        write a short, polite EMI reminder
 The model must GUESS:                       Context:     warm tone; help the customer pay on time
   who is writing? to whom?                  Constraints: at most 35 words · first name · exact
   how long? which facts?                                 amount and date · never mention penalties
   what tone? anything forbidden?            Format:      one plain-text message, no markdown

            same model · same input (Sneha Iyer · Rs 18,500 · due 5 November)
                                   │
                                   ▼
          five acceptance checks: ≤ 40 words · has "18,500" · has "5 November"
                                  · has "Sneha" · no penalty / late fee / legal
```
**Analogy — the capable temp.** You hire a very capable temp for one day. If all you say is "write a reminder", you will get *something*, but who knows what. A good brief says who they are writing as, what to do, what they must be told, the rules, and what the finished work looks like.

Notice one detail in the strong prompt: it asks for **at most 35 words** while the check allows **40**. Asking for a little less than the limit gives the model room to miss slightly and still pass.

### What the next cell does
1. Defines **`card(task)`**, which prints the task's title, scenario, the input, any material you may use, and the list of acceptance checks. `def` creates a function; it runs only when called. Inside, **f-strings** like `f"=== {task.id}: {task.title}"` put values into text: anything inside `{...}` is replaced by its value. `"\n".join(...)` glues a list of strings with line breaks between them. `if task.material:` is true only when the material dictionary is not empty.
2. Defines **`show(task, spec, label)`**: it calls `run_task` (one model call), prints the reply and the check table, and **returns two values**, `text, results`. Python returns them as a pair (a *tuple*), and `weak_out, weak_res = show(...)` unpacks the pair into two names.
3. Prints the DEMO card. This runs even without a key.
4. If `ready`, runs the weak prompt (`DEMO.weak_spec()`, a spec with only a task) and the strong one (`DEMO.reference`, all five parts). **2 model calls.** Without a key it prints "skipped".

**Predict first:** how many of the five checks will the weak prompt pass? Write a number down before you run.

In [ ]:
def card(task):
    print(f"=== {task.id}: {task.title}\n{task.scenario}\n")
    print("INPUT you are given:\n" + task.user_input + "\n")
    if task.material:
        print("MATERIAL you may put in your prompt:\n" + "\n".join(task.material.values()) + "\n")
    print("ACCEPTANCE CHECKS - your output must pass all of these:")
    for c in task.acceptance():
        print("   -", c)
    print()

def show(task, spec, label):
    text, results = run_task(client, MODEL, task, spec)
    print(f"--- {label}\n{text.strip()}\n{results_table(results)}\n")
    return text, results

card(DEMO)
if ready:                                                                     # the two model calls need a key
    weak_out, weak_res = show(DEMO, DEMO.weak_spec(), 'WEAK prompt: "' + DEMO.weak_prompt + '"')
    strong_out, strong_res = show(DEMO, DEMO.reference, "STRONG prompt (all five parts)")
else:
    print("skipped (no key in .env): the two model calls did not run")

### What you should see
First the card: the scenario, the three input lines, and the five acceptance checks. Then two blocks, one per prompt. Below is **an illustration, not a real run**: we wrote both replies ourselves and ran the *real* checks on them, so the format is exactly what you will see, but the words and the scores on your screen will be different.

```
--- WEAK prompt: "Write a reminder."
Reminder: your EMI payment is due soon. Please make sure you pay on time to avoid any late fees or penalties. Thank you.
  PASS  at most 40 words   (23 words)
  FAIL  includes the amount Rs 18,500
  FAIL  includes the due date
  FAIL  addresses the customer by name
  FAIL  does not threaten penalties or legal action   (found: ['late fee'])
  -> 1/5 checks passed

--- STRONG prompt (all five parts)
Hi Sneha, a friendly reminder that your EMI of Rs 18,500 is due on 5 November. Thank you for banking with us!
  PASS  at most 40 words   (22 words)
  PASS  includes the amount Rs 18,500
  PASS  includes the due date
  PASS  addresses the customer by name
  PASS  does not threaten penalties or legal action
  -> 5/5 checks passed
```

**How to read it:** each block shows the label, the model's reply, then one line per check. `PASS`/`FAIL` is the verdict; the bracket is the detail (here, the word count). The last line is the score.

| If you see... | It means... | Do this |
|---|---|---|
| the weak prompt passes several checks | a strong model got lucky on some of them | that is normal and worth saying: **lucky is not reliable**. Nothing in the prompt guaranteed it |
| the strong prompt fails a check | the model ignored a rule this time, or the check is stricter than it looks | read the detail in brackets; this is why we measure instead of trusting |
| `skipped (no key in .env)` | no key | read the illustration above, then run the under-the-hood cell below: it works offline |
| `RateLimitError` / `429` | the free plan's per-minute limit | wait 30 seconds and rerun this cell only |

### Under the hood: an acceptance check is just a small function
Checks sound formal, but each one is a name plus a few lines of Python. Here you rebuild two of the DEMO checks and confirm they give **exactly** the same verdict and detail as the real ones, on two replies we wrote by hand. No model call, so it runs without a key.

**What the next cell does:**
1. Imports the real `run_checks`, to compare against.
2. Defines `my_max_words(n)`. It returns a pair: the check's name, and a **`lambda`**, a one-line function with no name. `lambda t: (...)` means "given text `t`, return this". `t.split()` cuts the text at spaces, so `len(t.split())` counts words.
3. Defines `my_has_none_of(terms, label)`. Inside, a **list comprehension**, `[x for x in terms if x.lower() in t.lower()]`, builds the list of forbidden terms that appear in the text; `.lower()` makes the test case-insensitive. The check passes when that list is empty (`not found`). The detail uses a **conditional expression**: `A if condition else B`.
4. Runs both of your checks on a good reply and a bad one, and prints whether the real helper agrees.

In [ ]:
from prompt_tasks import run_checks                       # the real helper, to compare with

def my_max_words(n):
    return (f"at most {n} words", lambda t: (len(t.split()) <= n, f"{len(t.split())} words"))

def my_has_none_of(terms, label):
    def run(t):
        found = [x for x in terms if x.lower() in t.lower()]
        return (not found, f"found: {found}" if found else "")
    return (label, run)

good = "Hi Sneha, a friendly reminder: your EMI of Rs 18,500 is due on 5 November. Thank you!"
bad = good + " Please pay on time to avoid a late fee."
mine = [my_max_words(40), my_has_none_of(["penalty", "late fee", "legal"], "does not threaten penalties or legal action")]
for text in (good, bad):
    real = {r.name: (r.passed, r.detail) for r in run_checks(DEMO, text)}
    print(repr(text[-45:]))
    for name, fn in mine:
        ok, detail = fn(text)
        print(f"   {'PASS' if ok else 'FAIL'}  {name:<45} {detail!r:<22} real helper agrees: {real[name] == (ok, detail)}")

**What you should see:** for the good reply, two `PASS` lines (`17 words` and an empty detail). For the bad one, the word check still passes but the second says `FAIL ... "found: ['late fee']"`. Every line ends `real helper agrees: True`. `{detail!r}` prints the value with quotes (`!r` means "show it as Python would write it"), so an empty detail shows as `''` instead of nothing.

**The take-away:** a check is a promise you can test. Writing checks *first* tells you exactly what "done" means before you touch the prompt. And be honest about their limit: no check here can tell whether the reminder sounds kind.

### Check yourself
*Question 1 — Which acceptance checks did the weak prompt fail? For each one, which missing part of the prompt would have prevented it?*

> **Common mistake:** deciding which prompt is better by reading the replies and going with your feeling. Read the check table. If the weak prompt happened to pass, the honest conclusion is "it can work", not "it will work".

## 3. The parts, and where each one goes

### What and why
A prompt has five parts, plus optional examples. `PromptSpec` holds them and shows how they become the **messages list** you already know from Sessions 2 and 11. Standing rules (role, constraints, format) go in the **system** message. The job itself (task, context, the actual input) goes in the **user** message.

This section has no model call. You look at the prompt itself: what was missing from the weak one, and what the strong one *really* sends. Once you can print a prompt, nothing about it is magic.

### The idea: standing orders and the job slip
```
 PromptSpec (what you fill in)                 messages list (what is sent)
 ─────────────────────────────                 ──────────────────────────────────────────────
 role           ─────────────────────────┐
 constraints    ─────────────────────────┼───► [SYSTEM]  role
 output_format  ─────────────────────────┘               Rules:
                                                         - constraint 1
                                                         - constraint 2
                                                         Output format: ...
 examples [(in, out), ...]  ─────────────────► [USER] Input: in    [ASSISTANT] out   (one pair each)
 task           ─────────────────────────┐
 context        ─────────────────────────┼───► [USER]    Task: ...
 the input (user_input) ─────────────────┘               Context: ...
                                                         Input: ...
```
**Analogy.** The **system** message is the standing orders pinned to the wall: who you are, the rules, how finished work looks. They do not change from one job to the next. The **user** message is the job slip on today's desk: what to do, the papers for this job, and the item itself.

Why this split? Standing rules apply to every call, so they belong where the model treats them as the frame for the whole conversation. The facts change per request. Keeping them apart also makes the prompt easier to read, review and reuse: you can swap the input without touching the rules.

### What the next cell does
1. **`show_tips(DEMO.weak_spec())`** looks at each of the five parts and prints a tip for every empty one. The weak spec has only a task, so it lists four.
2. **`DEMO.reference.to_messages(DEMO.user_input)`** builds the real messages list for the strong prompt and this input.
3. The `for m in ...:` loop prints each message. `m['role']` reads the `"role"` key of the dictionary; `.upper()` writes it in capitals; the f-string puts it in square brackets, then the content below.

In [ ]:
print("Parts missing from the weak prompt:")
show_tips(DEMO.weak_spec())
print("\nWhat the strong prompt actually sends:\n")
for m in DEMO.reference.to_messages(DEMO.user_input):
    print(f"[{m['role'].upper()}]\n{m['content']}\n")

### What you should see
This cell does not call the model, so your output is **exactly** this (a real run):

```
Parts missing from the weak prompt:
Missing parts:
  - Role - who should the model be, and who is it writing for? ('You are a credit analyst writing for a credit committee.')
  - Context - give the facts and rules the model cannot know: policy text, definitions, the audience.
  - Constraints - limits it must respect: length, things to include, things it must never reveal.
  - Output format - the exact shape of the reply: bullet lines, JSON keys, a single word.

What the strong prompt actually sends:

[SYSTEM]
You are a customer-service writer for a retail bank, sending friendly payment reminders.

Rules:
- Use at most 35 words.
- Address the customer by first name.
- Include the exact amount and the due date.
- Never mention penalties, late fees or legal action.

Output format: One plain-text message. No subject line, no markdown.

[USER]
Task: Write a short, polite EMI reminder message.

Context:
The customer's details are in the input. The tone is warm and respectful: the aim is to help the customer pay on time, not to pressure them.

Input:
Customer: Sneha Iyer
EMI amount: Rs 18,500
Due date: 5 November
```

**How to read it:** the system message starts with the role, then a `Rules:` list (one line per constraint), then `Output format:`. The user message has `Task:`, `Context:` and `Input:` with the customer's three lines. Find each of the five parts. Nothing else is sent: no hidden memory of earlier sessions, no "understanding" of the bank. If it is not in this list, the model does not have it.

### Under the hood: build the messages list yourself
`to_messages` is short. Here is the same logic written out, and a check that it builds **exactly** the same list as the real helper for the weak prompt, the strong prompt, all three task reference prompts, and the 4-example classifier. Nothing is printed from the task references, so nothing is given away.

**What the next cell does:**
1. Builds the system text from the role, a `Rules:` block and the format. Each piece is added only if it is not empty (`.strip()` removes spaces at the ends, so a spec with `role=" "` counts as empty).
2. Defines a small inner function `user(text)` that builds `Task:` / `Context:` / `Input:`, skipping empty parts. `"\n\n".join(p for p in parts if p)` joins the non-empty pieces with a blank line between them.
3. Adds each example as a **user** turn (the example input, in the *same shape* as the real question) and an **assistant** turn (the correct answer). `msgs += [a, b]` adds two items to the list.
4. Compares your result with the real one for six specs, and prints the message roles of the few-shot classifier.

In [ ]:
from prompt_tasks import TASK1, TASK2, TASK3, classify_spec

def my_to_messages(spec, user_input):
    rules = [c.strip() for c in spec.constraints if c and c.strip()]
    system = [p for p in (spec.role.strip(), "Rules:\n" + "\n".join(f"- {c}" for c in rules) if rules else "",
                          "Output format: " + spec.output_format.strip() if spec.output_format.strip() else "") if p]
    def user(text):
        parts = ["Task: " + spec.task.strip() if spec.task.strip() else "",
                 "Context:\n" + spec.context.strip() if spec.context.strip() else "",
                 "Input:\n" + text.strip() if text.strip() else ""]
        return "\n\n".join(p for p in parts if p)
    msgs = [{"role": "system", "content": "\n\n".join(system)}] if system else []
    for ex_in, ex_out in spec.examples:
        msgs += [{"role": "user", "content": user(ex_in)}, {"role": "assistant", "content": ex_out}]
    return msgs + [{"role": "user", "content": user(user_input)}]

for name, spec, text in [("DEMO weak", DEMO.weak_spec(), DEMO.user_input), ("DEMO strong", DEMO.reference, DEMO.user_input),
                         ("TASK1 reference", TASK1.reference, TASK1.user_input), ("TASK2 reference", TASK2.reference, TASK2.user_input),
                         ("TASK3 reference", TASK3.reference, TASK3.user_input), ("classifier, 4 examples", classify_spec(4), "My card is blocked.")]:
    print(f"{name:<24}", "same" if my_to_messages(spec, text) == spec.to_messages(text) else "DIFFERENT")
print("\nroles sent by the 4-example classifier:", [m["role"] for m in classify_spec(4).to_messages("My card is blocked.")])

**What you should see:** `same` six times, then:
```
roles sent by the 4-example classifier: ['user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user']
```
The classifier has no role, constraints or format, so there is **no system message at all**: four example pairs, then the real question last. Change one character in `my_to_messages` (say, `"Rules:"` to `"Rule:"`) and rerun: some lines turn `DIFFERENT`. That is how you know the check is real.

| If you see... | It means... | Do this |
|---|---|---|
| `DIFFERENT` without editing anything | your course files are out of date | `git pull`, restart the kernel |
| `NameError: DEMO` | section 1 was not run | run section 1 first |

### Check yourself
*Question 2 — Why do the rules live in the system message and the facts in the user message? What would change if you moved the policy text from context into the system message for every call?*

> **Common mistake:** putting everything into one long user message ("You are... Write... Rules... Here is the data..."). It often works in a demo. But the rules then travel with the data, are rewritten for every request, and are easy to lose when someone changes the input code.

## 4. Zero-shot, one-shot, few-shot

### What and why
Same instruction every time: classify a bank customer's message as card, loan, account or other. Only the number of **worked examples** changes (0, 1, 4). The examples teach the bank's *house convention*: UPI problems are "account", servicing requests are "other". Two scores per run: did it pick the right label, and did it obey "reply with the single word only"?

This makes 24 short calls. Results vary by model; read your own numbers.

This little classifier matters beyond today. Session 15 reuses it to ask whether "thinking first" helps, and in Session 16 its one-line instruction becomes the **baseline** that your structured prompt must beat.

### The idea: show, don't just tell
```
 ZERO-SHOT (0)                  ONE-SHOT (1)                         FEW-SHOT (4)
 ─────────────────────          ──────────────────────────────       ──────────────────────────────────────
 [USER] Task: Classify...       [USER] Task... Input: Net banking    4 pairs: transfer pending -> account
        Input: <message>               transfer pending...                    lost debit card  -> card
                                [ASSISTANT] account                           no-dues cert.    -> loan
                                [USER] Task... Input: <message>               change mobile no -> other
                                                                     then [USER] Task... Input: <message>

 cheapest; fine when the        shows the format and the voice       best for teaching a house convention;
 task is obvious                                                     you pay for the examples on EVERY call
```
**Analogy — the new joiner.** A new colleague does not learn the bank's filing habits from the policy manual. They learn them from seeing three filled-in forms. "A UPI failure goes under *account*" is obvious to everyone at this bank and invisible to everyone outside it, including the model.

**Two scores, not one.** *Right label* asks "did it understand?". *Exact format* asks "did it obey 'single word only'?". A reply like "This is about a card." has the right label and the wrong format. Software reading the reply needs both. Examples often earn their cost on the second score: four replies of exactly one word show the shape more clearly than any instruction.

The eight test messages are **kept apart** from the four examples: no test message appears as an example (the test file checks this). Otherwise the score would be inflated, like marking your own homework. Session 16 makes this a strict rule.

### What the next cell does
1. If `ready`, prints a header. `{'examples':>9}` is an **f-string format spec**: `>9` means "right-align in 9 characters", so the columns line up.
2. **`for n in (0, 1, 4):`** loops over a tuple of three numbers. For each, `classify_spec(n)` builds the spec with the first `n` examples, and `score_classifier(...)` sends all eight test messages (**8 calls**) and returns a **dictionary** with `accuracy`, `exact_format`, `n` and `wrong`.
3. Prints one row: `r['accuracy']` reads a value from the dictionary by its key.
4. For each mistake, prints the message (first 58 characters: `text[:58]` is a **slice**), what the bank says, and what the model said. `!r` shows the values in quotes, so `None` (no label found at all) is easy to tell apart from a word.
5. Without a key, prints "skipped".

**Predict first:** which message will zero-shot most likely get wrong, and why?

In [ ]:
if ready:                                                                     # 24 model calls need a key
    print(f"{'examples':>9} {'right label':>12} {'exact format':>13}")
    for n in (0, 1, 4):
        r = score_classifier(client, MODEL, classify_spec(n))
        print(f"{n:>9} {r['accuracy']:>8}/{r['n']} {r['exact_format']:>9}/{r['n']}")
        for text, truth, got in r["wrong"]:
            print(f"            wrong: {text[:58]!r}  bank says {truth!r}, model said {got!r}")
else:
    print("skipped (no key in .env): the 24 classifier calls did not run")

### What you should see
A header and three rows, with any wrong answers listed under their row. Below is **an illustration, not a real run**: the layout is exact, the numbers are invented.

```
 examples  right label  exact format
        0        6/8         6/8
            wrong: 'Please update the nominee details on my savings account.'  bank says 'other', model said 'account'
            wrong: 'UPI payment of Rs 4,500 failed but the money was debited f'  bank says 'account', model said None
        1        7/8         8/8
            wrong: 'Please update the nominee details on my savings account.'  bank says 'other', model said 'account'
        4        8/8         8/8
```

**How to read it:** `6/8` under *right label* means six of eight labels matched the bank's convention. `6/8` under *exact format* means six replies were exactly one word. A `wrong:` line shows the message, the bank's label, and the model's label (`None` means no label word was found in the reply).

**Your numbers will be different, and that is the point of reading your own screen.** A strong model may already score well with zero examples. Look for three things:

| What to look at | What it tells you |
|---|---|
| Wrong answers on the "nominee" or "address" messages, or on UPI | the model's common sense disagrees with the bank's house convention; examples are the cure |
| *Exact format* lower than *right label* | it understood but chattered ("This is a card issue."); examples show the shape |
| All three rows at 8/8 | this model handles this task without examples. A **finding, not a failure**: you would now pay for four examples on every call for no gain |

If a cell stops with `RateLimitError` (429), you sent 24 calls too fast for the free plan's per-minute limit. Wait 30 seconds and rerun.

### Under the hood: how a reply becomes a label
`score_classifier` asks the model, then calls `read_label` to turn a free-text reply into a label and a yes/no for "exactly one word". Here you rebuild `read_label` and check it against the real one on seven replies. Then you score **eight replies we made up** (not a model run) to see how the two numbers are counted.

**What the next cell does:**
1. `re.sub(r"[^a-z ]", "", reply.lower())` deletes every character that is not a lowercase letter or a space (`re` is Python's regular-expression module; `[^...]` means "anything except"). `" Loan. "` becomes `"loan"`.
2. If what is left is exactly a label, the reply was exact: return `(label, True)`.
3. Otherwise, search for each label as a whole word (`\b` marks a word boundary) and return the first one found, with `False`.
4. Compares with the real `read_label` on seven replies, then scores the made-up replies against the eight test messages. **`zip(a, b)`** walks two lists side by side, giving one pair at a time.

In [ ]:
import re
from prompt_tasks import COMPLAINTS_TEST, LABELS, read_label

def my_read_label(reply):
    cleaned = re.sub(r"[^a-z ]", "", reply.lower()).strip()
    if cleaned in LABELS:
        return cleaned, True
    for lab in LABELS:
        if re.search(rf"\b{lab}\b", cleaned):
            return lab, False
    return None, False

for reply in ["card", " Loan. ", "**Account**", "I think this is a card problem.", "Category: other", "no idea", "the loan account"]:
    print(f"{reply!r:<34} -> {my_read_label(reply)}   same as real: {my_read_label(reply) == read_label(reply)}")

made_up = ["card", "Loan.", "account", "other", "This is about a card.", "loan", "account", "account"]   # invented replies, NOT a model run
right = exact = 0
for (text, truth), reply in zip(COMPLAINTS_TEST, made_up):
    label, was_exact = my_read_label(reply)
    right += label == truth
    exact += was_exact
print(f"\nmade-up replies: right label {right}/8, exact format {exact}/8")

**What you should see:** seven lines ending `same as real: True`. `"**Account**"` gives `('account', True)`: the stars are removed, so it still counts as exact. `"the loan account"` gives `('loan', False)`: two label words, and the first one in the `LABELS` list wins. That is an honest limit of this simple reader: a chatty reply can be scored as the wrong label. It is one more reason to demand "single word only".

Then: `made-up replies: right label 7/8, exact format 7/8`. Reply 5 had the right label in the wrong format; reply 7 said `account` where the bank says `other` (the nominee message). `right += label == truth` works because in Python `True` counts as 1 and `False` as 0.

### Check yourself
*Question 3 — Did the examples change the label, the format, or both? A strong model may already score well with zero examples. When would you still pay for the extra tokens of few-shot on every call?*

> **Common mistake:** concluding "few-shot is always better" (or "never needed") from eight messages. Eight is a demonstration, not a measurement. Session 16 measures on twenty held-out messages, and Session 17 builds a proper golden set.

## 5. Your turn: three real prompts

### What and why
For each task: read the card, look at what the weak prompt produces, then fill in the parts in `my_spec_N` and rerun until every acceptance check passes. Change **one thing at a time** so you learn which change helped. Do not peek at `prompt_tasks.py`: the instructor's reference prompts are in there, and the learning is in trying first.

These three tasks are the failures from the story in A1: a summary that ignores policy, an SMS that leaks a secret, and triage output that breaks software. Each one needs a different part of the prompt most.

### The idea: the improvement loop
```
   read the card ──► run the weak prompt ──► read which checks FAIL
                                                   │
          ┌────────────────────────────────────────┘
          ▼
   pick ONE failing check ──► ask "which part would have prevented it?" ──► fill in that part
          ▲                                                                        │
          │                                                                        ▼
          └──────────── not all PASS ◄──────────── rerun your cell, read the checks
                                                         │
                                                    all PASS ──► go on to the next task
```
**Read the check's name: it tells you which part is missing.** A useful map:

| A failing check about... | Usually needs... |
|---|---|
| length (`at most N words / characters`) | a **constraint**, a little stricter than the check |
| a fact that must appear (a name, a number, a reference) | a **constraint** ("include ... exactly as given") |
| something that must never appear | a **constraint** that names what is forbidden, plus **context** saying why |
| a decision a policy decides (`recommends 'refer'`) | the policy in the **context**, and a constraint to apply it exactly |
| shape (`exactly 3 bullet lines`, `JSON object only`, exact keys) | the **output format**, written precisely |

**Analogy — debugging with one change at a time.** If you change five things and the checks pass, you do not know which change helped, or whether two of them were useless. Change one, rerun, read. It is slower for ten minutes and much faster for the rest of your career.

### How each task's code works
Every task has two cells.
- **The first** prints the card and, if `ready`, runs the task's weak prompt once (1 call). Run it once; you do not need to rerun it.
- **The second** is yours. `PromptSpec(role="", task=..., ...)` creates a spec using **keyword arguments**: each value is given by name, so the order does not matter and the code reads like a form. `constraints=[]` is an empty **list**: add one string per rule, separated by commas, e.g. `["Use at most 60 words.", "Never ..."]`. The cell prints `show_tips` (which parts are still empty; this works without a key), then, if `ready`, sends your prompt (1 call per run) and prints the reply and the checks.

The material you may use is in `TASKn.material`, a dictionary. You can paste it into a part directly, e.g. `context=TASK1.material["policy"]`.

Each rerun of your cell is a new model call (this session has no cache), so read the checks carefully before rerunning.

### Task 1 — a summary a credit committee can trust

**What and why.** A credit committee reads dozens of summaries a day. It wants three lines and a recommendation it can trust. The applicant's data is in the input; the bank's lending policy is in `TASK1.material["policy"]`. The model has never seen this policy. However capable it is, it cannot apply a rule it was never shown.

**What the next cell does:** prints the card (scenario, input, the policy, the four checks), then runs the weak prompt "Summarise this applicant." (1 call, or "skipped").

**Predict first:** which of the four checks can the weak prompt *possibly* pass, given that it never saw the policy?

In [ ]:
card(TASK1)
if ready:
    w1 = show(TASK1, TASK1.weak_spec(), 'WEAK prompt: "' + TASK1.weak_prompt + '"')
else:
    print("skipped (no key in .env): the weak prompt was not sent")

**What you should see.** The card lists four checks: at most 80 words, exactly 3 bullet lines, states the total EMI-to-income ratio (about 33–34%), and recommends "refer". Then the weak reply. Here is **an illustration, not a real run** (a generic reply we wrote, scored by the real checks):

```
--- WEAK prompt: "Summarise this applicant."
Kavya is a 34 year old engineer with a good credit score who is asking for a loan and seems a solid candidate.
  PASS  at most 80 words   (23 words)
  FAIL  exactly 3 bullet lines   (0 bullet lines)
  FAIL  states the total EMI-to-income ratio (about 33-34%)
  FAIL  recommends 'refer', as the policy requires
  -> 1/4 checks passed
```

**Now your version.** In the next cell, fill in one part at a time. Hints, not answers:
- The ratio check needs a number the model must *calculate*. Does it know what "total EMIs" means here (existing plus proposed)? If not, tell it how.
- The policy decides the recommendation. Where does the policy go?
- "Exactly three bullet lines" is a shape. Say what each line contains and how it starts.

In [ ]:
my_spec_1 = PromptSpec(
    role="",              # YOUR TURN: who is the model, and who is it writing for?
    task=TASK1.weak_prompt,   # sharpen this: one sentence, starts with a verb
    context="",           # facts and rules the model cannot know (the lending policy is in TASK1.material["policy"])
    constraints=[],       # limits: length, what must be included, what must never be revealed
    output_format="",     # the exact shape of the reply
)
show_tips(my_spec_1)
if ready:
    out1, res1 = show(TASK1, my_spec_1, "YOUR prompt, version 1")
else:
    print("skipped (no key in .env): your prompt was not sent")

**What you should see.** First `Missing parts:` with a tip for each empty part (or `All five parts present.`). Then your reply and its checks. Aim for `-> 4/4 checks passed`.

| If you see... | It means... | Do this |
|---|---|---|
| `FAIL exactly 3 bullet lines (0 bullet lines)` | the lines do not start with `- `, `* `, `•` or `1.` | say in the format how each line must start |
| `FAIL ... 33-34%` but a ratio is shown | it used only the new EMI, or rounded oddly | state the formula in a constraint or the context |
| `FAIL recommends 'refer'` | the policy is missing, or the model added its own opinion | put the policy in context; add "apply the policy exactly" |
| `NameError: name 'PromptSpec' is not defined` | section 1 was not run in this kernel | run section 1 |

> **Common mistake:** writing the policy *summary* in your own words. Paste the policy as it is. Your paraphrase may drop exactly the rule that decides this applicant.

### Task 2 — an SMS that must not leak

**What and why.** The customer reads this on a phone, in a hurry, possibly upset. Some of the facts you hold must never reach them: the input contains an *internal, confidential* risk score and cutoff. The model sees everything in the input. Unless the prompt says otherwise, a helpful model may "explain" the decision using exactly those numbers.

**What the next cell does:** prints the card (six checks, including "does NOT reveal the confidential internal reason") and runs the weak prompt (1 call, or "skipped").

**Predict first:** will the weak prompt mention 612 or 650? Why might a model think that is helpful?

In [ ]:
card(TASK2)
if ready:
    w2 = show(TASK2, TASK2.weak_spec(), 'WEAK prompt: "' + TASK2.weak_prompt + '"')
else:
    print("skipped (no key in .env): the weak prompt was not sent")

**What you should see.** Six checks: at most 300 characters, includes the reference `APP-20931`, includes the helpline, mentions the 90-day reapply period, addresses Rohan by name, and does not reveal the internal reason. A weak reply often fails the length (it writes an email, not an SMS) and may leak the numbers. Read the detail in brackets: `found: ['612', 'risk score']` names exactly what leaked.

**Now your version.** Hints, not answers:
- An SMS has a hard length. Say it in characters, a bit under the check.
- "Never reveal" works best when you name *what* must never appear, and say why (context).
- Facts that must appear exactly (reference, helpline) need saying: "exactly as given".

In [ ]:
my_spec_2 = PromptSpec(
    role="",              # YOUR TURN: who is the model, and who is it writing for?
    task=TASK2.weak_prompt,   # sharpen this: one sentence, starts with a verb
    context="",           # facts and rules the model cannot know (and what must never be shared)
    constraints=[],       # limits: length, what must be included, what must never be revealed
    output_format="",     # the exact shape of the reply
)
show_tips(my_spec_2)
if ready:
    out2, res2 = show(TASK2, my_spec_2, "YOUR prompt, version 1")
else:
    print("skipped (no key in .env): your prompt was not sent")

**What you should see.** Aim for `-> 6/6 checks passed`.

| If you see... | It means... | Do this |
|---|---|---|
| `FAIL at most 300 characters (412 characters)` | it wrote a letter | a character limit in constraints, and "one plain-text SMS" in the format |
| `FAIL does NOT reveal ... (found: ['cutoff'])` | the confidential line leaked, perhaps re-worded | name the forbidden items (scores, cutoffs, the internal reason) |
| `FAIL includes the helpline number` | it wrote "call our helpline" | "include the helpline number exactly as given" |

> **Common mistake:** deleting the confidential line from the input to "fix" the leak. In a real system you do not control the input: the data arrives with everything in it. The prompt must handle it. (Session 18 shows why the prompt alone is not enough either, and adds an output check.)

### Task 3 — JSON that a ticketing system can read

**What and why.** The output goes straight into a ticketing system. A single stray sentence breaks the integration. The category and urgency definitions are in `TASK3.material["categories"]`. Note the house convention again: UPI and statements are *account*; address or nominee changes are *other*.

**What the next cell does:** prints the card (four checks: bare JSON only, exactly two keys, the right category, the right urgency) and runs the weak prompt "What is this complaint about?" (1 call, or "skipped").

**Predict first:** the weak prompt asks a question. What shape of answer does a question invite?

In [ ]:
card(TASK3)
if ready:
    w3 = show(TASK3, TASK3.weak_spec(), 'WEAK prompt: "' + TASK3.weak_prompt + '"')
else:
    print("skipped (no key in .env): the weak prompt was not sent")

**What you should see.** The weak prompt almost always fails all four checks: it answers in sentences. The first check, `reply is a JSON object only (no prose, no code fences)`, uses Python's `json.loads` on the whole reply. Anything around the JSON, even a polite "Here is the JSON:", or a Markdown fence (three backticks then `json`), makes it fail, exactly as it would break the ticketing system.

**Now your version.** Hints, not answers:
- Name the exact keys, and the allowed values for each.
- Say what must *not* surround the JSON. Models love code fences.
- The definitions decide "high" versus "medium". Give them.

In [ ]:
my_spec_3 = PromptSpec(
    role="",              # YOUR TURN: who is the model, and who is it writing for?
    task=TASK3.weak_prompt,   # sharpen this: one sentence, starts with a verb
    context="",           # facts and rules the model cannot know (the category and urgency definitions are in TASK3.material)
    constraints=[],       # limits: length, what must be included, what must never be revealed
    output_format="",     # the exact shape of the reply
)
show_tips(my_spec_3)
if ready:
    out3, res3 = show(TASK3, my_spec_3, "YOUR prompt, version 1")
else:
    print("skipped (no key in .env): your prompt was not sent")

**What you should see.** Aim for `-> 4/4 checks passed`. A passing reply is one line like `{"category": "...", "urgency": "..."}` and nothing else.

| If you see... | It means... | Do this |
|---|---|---|
| `FAIL reply is a JSON object only (did not parse as bare JSON)` and the reply has a fence | it wrapped the JSON in a code block | "no markdown code fences" in the output format |
| `FAIL has exactly the keys ...` | an extra key such as `"reason"` | "exactly the keys ... and nothing else"; "do not explain" |
| `FAIL urgency ... (urgency = 'medium')` | it did not see the 24-hour rule | give the urgency definitions in the context |

<details><summary><b>▸ Read more: asking for JSON versus guaranteeing it</b></summary>

Today you *ask* for JSON in the prompt and *check* it with code. In Session 13 you met the stronger tool: schema-enforced output, where the provider constrains the model to produce JSON that matches a schema. Prompting is still worth learning: not every model or provider supports strict schemas, and the definitions (what "high" urgency means) still have to be written in words. In production you do both: a clear prompt, a schema where available, and a check that never trusts either blindly.
</details>

### Check yourself
*Question 4 — Which task needed the most iterations, and which single constraint or format line made the biggest difference? Which acceptance check would you add that we did not write?*

> **Common mistake:** writing a huge prompt at once, passing every check, and moving on. You will not know which lines matter, and next month you will be afraid to delete any of them. The library in section 6 only helps if each version records one understandable change.

## 6. Your prompt library — a versioned asset

### What and why
A prompt that works today will be edited next month by someone who forgot why it says what it says. So keep each version, the output it produced, and which checks it passed, in a file that lives in Git: `prompts/library.md` in **your work repository** (not the course repo). Never edit an old entry — add a new version. Git then gives you the history for free.

First, point the notebook at your cloned work repository.

### The idea: two folders, two jobs
```
 Documents\
   genai-agents-course\    <- the COURSE repo. You only ever  git pull  into it.
   genai-course-work\      <- YOUR repo. You  git add / commit / push  from it.   (NOT inside the one above)
     .gitignore            <- lists .env, so your key can never be committed
     README.md
     prompts\
       library.md          <- one entry per prompt version, newest at the bottom
```
**Analogy.** The course repository is a library book the instructor keeps updating: you read the new pages (`git pull`) but you do not write in it. Your work repository is your own notebook: you write in it, save versions (`commit`) and hand it in (`push`). Each folder has a hidden `.git` folder holding its history; if you put one inside the other, Git cannot tell which history you mean.

A library entry looks like this (the format `save_to_library` writes):
```
## T2-sms-declined | v1 | 2026-10-12

**Purpose:** Tell a customer their application was declined, by SMS
**Result:** 6/6 checks passed

### Prompt
- **Role:** ...   - **Task:** ...   - **Context:** ...   - **Constraints:** ...   - **Output format:** ...
### Output
> Hi Rohan, ...
### Checks
- [x] at most 300 characters (182 characters)
```
(The output and the numbers in this sample are only to show the layout.)

### What the next cell does
1. **`Path.home() / "Documents" / "genai-course-work"`** builds a path. `Path.home()` is your user folder (`C:\Users\you` on Windows); the **`/` operator** joins path pieces, and `pathlib` uses the right separator for your system. This is the same folder as `~\Documents\genai-course-work` in PowerShell.
2. Prints the path and whether it **exists**.
3. If it exists, `init_work_repo` creates `.gitignore`, `README.md` and `prompts/library.md` **only if they are missing** (it never overwrites your files) and returns the names it created. `init_work_repo(...) or "nothing ..."` uses **`or`** as a fallback: an empty list counts as false, so the message after `or` is printed instead.
4. If the folder does not exist, it tells you to clone it first.

In [ ]:
WORK_REPO = Path.home() / "Documents" / "genai-course-work"      # edit if you cloned it somewhere else
print("work repo:", WORK_REPO, "| exists:", WORK_REPO.exists())
if WORK_REPO.exists():
    print("created:", init_work_repo(WORK_REPO) or "nothing (all files were already there)")
else:
    print("Clone your work repository first (module02/work_repo_steps.md), then rerun this cell.")

### What you should see
The first time, something like:
```
work repo: C:\Users\you\Documents\genai-course-work | exists: True
created: ['.gitignore', 'prompts/library.md']
```
(`README.md` is usually missing from this list because GitHub created one when you ticked "Add a README file".) The second time: `created: nothing (all files were already there)`.

| If you see... | It means... | Do this |
|---|---|---|
| `exists: False` | you have not cloned it, or you cloned it somewhere else | clone it (`module02/work_repo_steps.md`, step 3), or edit `WORK_REPO` to the real path (next row) |
| `exists: False` but you did clone it | the path differs (another drive, a OneDrive folder) | in PowerShell, `cd` into your clone and run `pwd`; copy the path; set `WORK_REPO = Path(r"C:\...\genai-course-work")` |
| the path ends in `genai-agents-course` | you pointed it at the **course** repo | point it at `genai-course-work`; never write into the course repo |

`r"..."` is a **raw string**: backslashes are kept as they are, which is what you want for Windows paths.

### Save your three prompts (once)
When all three of your prompts pass their checks, set `SAVE = True` and run the next cell **once**. (It stays off by default so that "Run All" never saves half-finished prompts.) If you improve a prompt later, bump `VERSION` and run again: a new entry is added, the old one stays.

**What the next cell does:**
1. `SAVE = False` and `VERSION = 1` are the two switches you change.
2. `if SAVE and WORK_REPO.exists():` both must be true.
3. The loop walks a **tuple of tuples**: each inner tuple is `(task, spec, out, res)`, and `for task, spec, out, res in (...)` unpacks the four values on each pass.
4. `save_to_library(...)` **appends** one entry per task, with today's date, your spec's five parts, the reply, and the ticked or unticked checks.

In [ ]:
SAVE = False      # set to True when you are happy with all three prompts
VERSION = 1
if SAVE and WORK_REPO.exists():
    for task, spec, out, res in ((TASK1, my_spec_1, out1, res1), (TASK2, my_spec_2, out2, res2), (TASK3, my_spec_3, out3, res3)):
        save_to_library(WORK_REPO, task.id, VERSION, spec, out, res, purpose=task.title)
    print("saved 3 entries to", WORK_REPO / "prompts" / "library.md")
else:
    print("not saved (SAVE is False, or the work repo folder is missing)")

### What you should see
With `SAVE = False` (the default): `not saved (SAVE is False, or the work repo folder is missing)`. After you set `SAVE = True` and run once: `saved 3 entries to C:\Users\you\Documents\genai-course-work\prompts\library.md`. Open that file in VS Code and find your three entries.

| If you see... | It means... | Do this |
|---|---|---|
| `NameError: name 'out1' is not defined` (or `out2`, `out3`) | a task cell did not run, or printed "skipped" (no key) | run the three task cells with a key, then this cell |
| six entries instead of three | you ran the save cell twice with `VERSION = 1` | harmless; never delete entries. Next time bump `VERSION` first |
| an entry with `0/4 checks passed` | you saved before the prompt passed | fix the prompt, set `VERSION = 2`, save again. Keep the old entry: it is honest history |

<details><summary><b>▸ Read more: why keep failed versions at all?</b></summary>

Because the failures are the explanation. "Why does our SMS prompt say *never mention scores or cutoffs*?" The answer is version 1, which leaked the score. Without it, someone "simplifies" the prompt and the leak returns. This is the same reason engineers keep Git history and write commit messages. In Session 15 you add version 2 entries, and a check confirms version 1 is still there. Later (Session 54) teams run two prompt versions side by side on real traffic before switching, which only works if both versions are recorded.
</details>

### Check yourself
*Why does the save cell append a new entry instead of updating the old one? What would you lose if it overwrote?*

> **Common mistake:** setting `WORK_REPO` to the course folder, or saving into a `prompts` folder you created inside `genai-agents-course`. The course repo is pull-only; your next `git pull` may clash with it, and your instructor will never see it.

## 7. Commit and push — your first submission

### What and why
Your library exists on your laptop. Your instructor cannot see your laptop. Two Git steps turn it into a submission: a **commit** records a labelled snapshot locally; a **push** sends it to GitHub. Then a self-check reads your repository and tells you, in plain English, whether everything really arrived.

### The idea
```
  library.md ──git add .──► staged ──git commit -m "..."──► snapshot on your laptop ──git push──► GitHub
                            (chosen for the snapshot)        (nobody else sees it yet)          (instructor can read it)
```
**Analogy.** `git add` puts the pages in the envelope. `git commit` seals it and writes a label on it. `git push` posts it. A sealed envelope on your desk has not been handed in.

In a terminal (not in this notebook), inside your **work** repository:

```powershell
cd ~\Documents\genai-course-work
git add .
git commit -m "Session 14: prompt library v1"
git push -u origin main        # the first time; afterwards just: git push
```
The first push opens a browser window to sign in to GitHub. Then run the self-check below. It reads your repository and tells you, in plain English, what is missing and the exact command to fix it.

**What each command does:**
1. `cd ~\Documents\genai-course-work` moves into **your** repo. Most Git trouble this term comes from running a command in the wrong one of the two folders.
2. `git add .` stages every changed file in this folder (the `.` means "here"). `.gitignore` keeps `.env` out.
3. `git commit -m "..."` records the snapshot with a message that says what changed.
4. `git push -u origin main` sends the `main` branch to the remote called `origin`. `-u` remembers the link, so next time `git push` alone is enough.

Run `git status` before and after: it tells you what is staged, what is not, and whether you are ahead of GitHub.

### What the next cell does
1. **`check_work_repo(WORK_REPO)`** runs nine checks on your repository: the folder exists, it is a Git repository (cloned, not a zip), Git knows your name and email, `origin` points at GitHub, the library has entries for three different tasks, `.gitignore` lists `.env`, `.env` is not tracked, nothing is uncommitted, and everything is pushed.
2. For each result, prints `PASS` or `FAIL` using a **conditional expression**: `("PASS  " if r.passed else "FAIL  ")`.
3. For each failure, prints the **fix**: the exact command to run.

In [ ]:
for r in check_work_repo(WORK_REPO):
    print(("PASS  " if r.passed else "FAIL  ") + r.name)
    if not r.passed and r.detail:
        print("        fix:", r.detail)

### What you should see
When everything is done, nine lines, all `PASS`:
```
PASS  work repo folder exists
PASS  it is a Git repository (cloned, not downloaded as a zip)
PASS  Git knows your name and email
PASS  a remote called origin points at GitHub
PASS  prompts/library.md has entries for 3 different tasks
PASS  .gitignore lists .env
PASS  your .env is NOT tracked by Git
PASS  nothing left uncommitted
PASS  everything is pushed to GitHub
```
If the folder does not exist yet (for example, when running without a work repo), you see one line, `FAIL  work repo folder exists`, with the `git clone` fix. That is the self-check doing its job, not a crash.

| If you see... | It means... | Do this |
|---|---|---|
| `FAIL  Git knows your name and email` | step 1 of the work-repo steps was not done | `git config --global user.name "Your Name"` and `git config --global user.email "you@example.com"` |
| `FAIL  prompts/library.md has entries for 3 different tasks` | `SAVE` was never `True`, or a task was not saved | rerun the task cells, set `SAVE = True`, run the save cell once |
| `FAIL  nothing left uncommitted` | you changed files after the commit | `git add .` then `git commit -m "..."` |
| `FAIL  everything is pushed to GitHub` (`no upstream set`) | first push not done | `git push -u origin main` |
| `FAIL  your .env is NOT tracked by Git` | your secrets file was committed to a **public** repository | run the printed fix, push, and **create a new API key**: treat the old one as public |
| `Repository not found` (in the terminal, on push) | wrong URL, or signed in to another GitHub account | `git remote -v`; clear GitHub entries in Windows Credential Manager; push again |

The full table of Git errors, with every fix, is in `module02/work_repo_steps.md`.

### Check yourself
*You committed but did not push. Which self-check lines fail, and what does your instructor see on GitHub?*

> **Common mistake:** running `git add`, `commit` and `push` in the **course** folder. You will get an error on push (you cannot push to the instructor's repository), and your library is not in it anyway. `cd ~\Documents\genai-course-work` first.

## 8. Reflection
Answer questions 1-4 above, one or two lines each, and write one line on what was hardest about the Git steps.

1.
2.
3.
4.

Checkpoint: every line of the self-check above says `PASS`, and `uv run pytest tests/test_s14.py` is green in the course repository. Then submit your repository link through the form.

---
# Part C · Wrap-up

## C1. Summary
- **The same model gives very different results for different words.** A weak prompt makes the model guess; a strong prompt removes the guessing.
- **A prompt has five parts:** role, task, context, constraints, output format, plus optional examples. A missing part usually shows up as a specific failing check.
- **Standing rules go in the system message; the job goes in the user message;** examples become earlier user/assistant turns in the same shape as the real question.
- **Few-shot examples teach house conventions** that the model cannot guess. They cost tokens on every call, so measure whether they help.
- **"Better" needs a number.** Write acceptance checks first, change one part at a time, and read the checks. Checks catch what a machine can catch, not tone or truth.
- **A prompt is an asset.** Every version, its output and its checks go into `prompts/library.md`. Never edit an old entry; add a new version.
- **Your work repository is where you push; the course repository is where you pull.** The self-check proves your submission arrived.

## C2. The words again

| Term | One-line meaning |
|---|---|
| Prompt | Everything sent for one request: instructions, facts, examples and the input |
| System / user message | Standing rules for every call / this request's job and input |
| Role | Who the model is and who it writes for |
| Task | What to do, in one sentence starting with a verb |
| Context | Facts and rules the model cannot know unless you supply them |
| Constraints | Limits on length, content, and what must never appear |
| Output format | The exact shape of the reply |
| Zero / one / few-shot | No, one, or several worked examples in the prompt |
| House convention | A rule obvious inside your organisation, invisible to the model |
| Acceptance check | A few lines of code that pass or fail an output |
| Baseline | The simple version you compare against |
| Prompt library / versioning | Every prompt version with its output and checks; never overwrite, add v2 |
| Work repository | Your own GitHub repository, where you push |
| Commit / push / origin | A local snapshot / sending it to GitHub / the GitHub copy you are linked to |
| `.gitignore` / tracked | Files Git leaves alone / files Git records in commits |

## C3. Interview practice
Answer aloud first, then read the model answer. Each answer has a definition, an example from this lab, and one honest limit.

**1. What makes a good prompt? Walk me through its parts.**
*Model answer:* I think of five parts: the role (who the model is and for whom it writes), the task (one verb-led sentence), the context (facts and rules it cannot know), the constraints (length, must-include, must-never), and the output format (the exact shape), plus examples when there is a house convention. In my lab, "Write a reminder." gave the model nothing to go on, while the five-part version stated the word limit, the facts to include and that penalties must never be mentioned, and every check could then be tested. The limit: structure makes the right answer much more likely, not certain, so I always pair a prompt with checks.

**2. What goes in the system message and what goes in the user message?**
*Model answer:* Standing rules that apply to every request (role, constraints, format) go in the system message; the job for this request (task, context, input) goes in the user message, and few-shot examples become earlier user/assistant turns. In my lab, I printed the messages list for the EMI reminder: the system message held the role, a rules list and the format; the user message held the task and the customer's three lines. The limit: the system message is not a security boundary. A determined user can still try to override it, which is why defensive prompting and output checks exist.

**3. When would you use few-shot instead of zero-shot, and what does it cost?**
*Model answer:* Few-shot means putting worked input/output examples in the prompt. I use it when the task depends on a convention the model cannot guess, or when the output format must be exact. In my lab, the bank files UPI failures under "account" and nominee changes under "other"; I scored the classifier with 0, 1 and 4 examples on eight messages and looked at both the label and the single-word format. The cost is tokens on every call and the risk of the model copying the examples too closely; and eight messages is too few to be sure, so I would confirm on a larger held-out set.

**4. How do you know that a prompt change actually made things better?**
*Model answer:* I define "better" before I change anything, as acceptance checks: small pieces of code that pass or fail an output. Then I change one part at a time and rerun. In my lab, the declined-SMS task had six checks, including one that fails if the internal score or cutoff appears; the reply detail told me exactly what leaked, so I knew which constraint to add. The limit: checks only see what code can see. They cannot judge tone or factual truth, and one passing run is not proof. That is why Session 17 builds golden sets and judges.

**5. How do you manage prompts in a team?**
*Model answer:* As versioned assets in Git: each version is recorded with its purpose, its parts, the output it produced and which checks passed, and old versions are never edited, only superseded. In my lab, I kept `prompts/library.md` in my own repository, appended one entry per task with `save_to_library`, committed, pushed, and ran a self-check that confirmed the push and that `.env` was not tracked. The limit: a Markdown library is a start, not a full system. At team scale you also want reviews, automated tests on every change and the ability to roll back, which later sessions add.

## C4. Troubleshooting

| You see | It means | Fix |
|---|---|---|
| `ImportError ... run 'git pull'` (section 1) | a course file is missing | `git pull`, `uv sync`, **Kernel → Restart** |
| `skipped (no key in .env)` everywhere | no key, or the placeholder `paste_your_...` | paste your key in `.env` after `API_KEY=`, restart the kernel |
| `AuthenticationError` / `401` | the key is wrong or revoked | create a new key in the Groq console, paste it, restart the kernel |
| `RateLimitError` / `429` | too many tokens in a minute on the free plan (section 4 most often) | wait 30 seconds; rerun only that cell |
| `NameError: name 'client' is not defined` (or `card`, `show`) | an earlier cell was not run in this kernel | run from section 1, in order |
| `NameError: name 'out1' is not defined` in the save cell | a task cell did not run or was skipped | run the task cells with a key first |
| the weak prompt passes several checks | a strong model got lucky | expected: lucky is not reliable. Move on |
| a JSON reply looks right but fails `JSON object only` | it is wrapped in a code fence or has a sentence around it | forbid fences and prose in the output format |
| `exists: False` for the work repo | not cloned, or cloned elsewhere | clone it, or set `WORK_REPO = Path(r"<your path>")` |
| `Author identity unknown` (terminal) | Git does not know who you are | the two `git config --global user...` lines |
| `fatal: not a git repository` | wrong folder | `cd ~\Documents\genai-course-work` |
| `Updates were rejected ... fetch first` | GitHub has a change you don't (e.g. a README edited on the website) | `git pull`, then `git push`; if vim opens: `Esc`, `:wq`, `Enter` |
| self-check: `.env` is tracked | the secrets file was committed | run the printed fix, push, **create a new API key** |
| `pytest tests/test_s14.py` shows `4 skipped` | no key: the live reference-prompt tests did not run | that is expected offline; with a key they run. A skip is not a pass |

## C5. What comes next
**Session 15 — Reasoning prompts.** Should the model "think step by step" before it answers? You will measure accuracy *and* cost on a loan-eligibility checker, reuse today's complaint classifier, and add **version 2** entries to your library (version 1 must still be there).

**Keep from today:**
- your work repository, cloned at `Documents\genai-course-work`, with `prompts/library.md` pushed;
- the five-part habit and the "one change at a time" loop;
- the one-line classifier instruction from section 4: Session 16 uses it as the baseline your structured prompt must beat.

**Before Session 15:** library pushed, self-check all `PASS`, repository link submitted. Bring one prompt that still fails a check, and the check it fails.